# F04B INWIT — FFmpeg Finishing
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Appliquer la vitesse validée par F04A, produire `youtube_short.mp4` ou `youtube_long.mp4`.

**Entrées** :
- `F04_INWIT/IN/video_render.mp4` (produit par F03)
- `F04_INWIT/OUT/speed_validated.json` (produit par F04A)
- `F04_INWIT/OUT/plan_de_vol.json` (produit par F04A)

**Sorties** :
- `F04_INWIT/OUT/youtube_short.mp4` si format=vertical
- `F04_INWIT/OUT/youtube_long.mp4` si format=horizontal

---
### Logique setpts conditionnel V3
| playback_speed | Action FFmpeg |
|---|---|
| `1.0` | `-c copy` — stream copy, aucun re-encode, qualité bit-for-bit |
| `≠ 1.0` | `-vf setpts=PTS/{speed}` + `-af atempo={speed}` si audio |

---
### Avant de lancer
1. Monte ton Google Drive (cellule 1)
2. Vérifie que F04A a bien généré `speed_validated.json` dans `F04_INWIT/OUT/`
3. Lance les cellules dans l'ordre
4. Télécharge le fichier `youtube_*.mp4` depuis Google Drive pour upload YouTube

In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

In [ ]:
# CELLULE 2 — Configuration
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
print(f'Drive base : {DRIVE_BASE}')

In [ ]:
# CELLULE 3 — Vérification FFmpeg + aperçu de la vitesse figée
import subprocess, json
from pathlib import Path

# FFmpeg disponible sur Colab par défaut
r = subprocess.run(['ffmpeg', '-version'], capture_output=True)
if r.returncode == 0:
    ver = r.stdout.decode().split('\n')[0]
    print(f'FFmpeg OK : {ver}')
else:
    print('FFmpeg absent — installation...')
    subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=True)
    print('FFmpeg installé.')

# Aperçu vitesse figée
speed_file = Path(DRIVE_BASE) / 'F04_INWIT' / 'OUT' / 'speed_validated.json'
if speed_file.exists():
    with open(speed_file) as f:
        sv = json.load(f)
    speed = sv['playback_speed']
    mode  = 'stream copy (aucun re-encode)' if speed == 1.0 else f're-encode (setpts=PTS/{speed})'
    print(f'Vitesse figée : {speed}x  →  {mode}')
else:
    print('AVERTISSEMENT : speed_validated.json introuvable.')
    print('→ Lancer F04A (cellule 3 + FIGER LA VITESSE) avant cette cellule')

In [ ]:
# CELLULE 4 — Lancement F04B INWIT (FFmpeg Finishing)
import shutil, subprocess, sys
from pathlib import Path

script_src = Path(DRIVE_BASE) / 'F04_INWIT' / 'CODEBASE' / 'drn_f04b_inwit.py'
script_dst = Path('/content/drn_f04b_inwit.py')
shutil.copy2(script_src, script_dst)
print(f'Script copié : {script_dst}')

result = subprocess.run(
    [sys.executable, str(script_dst), '--drive-base', DRIVE_BASE],
    capture_output=False
)

if result.returncode == 0:
    print('\n✓ F04B INWIT — FINISHING OK')
    print('→ Récupérer le fichier youtube_*.mp4 dans Google Drive pour upload YouTube')
else:
    print('\n✗ F04B INWIT — FINISHING FAIL — corriger les erreurs FFmpeg ci-dessus')

In [ ]:
# CELLULE 5 — Transit CRS_CUSTOS (check-in F04B — SCELLEMENT FINAL)
# À lancer UNIQUEMENT si la cellule 4 s'est terminée avec FINISHING OK
import shutil, subprocess, sys, json
from pathlib import Path

# Vérification : youtube_*.mp4 doit exister
out_dir = Path(DRIVE_BASE) / 'F04_INWIT' / 'OUT'
out_short = out_dir / 'youtube_short.mp4'
out_long  = out_dir / 'youtube_long.mp4'
out_file  = out_short if out_short.exists() else (out_long if out_long.exists() else None)

if out_file is None:
    print('ERREUR : youtube_short.mp4 / youtube_long.mp4 introuvable.')
    print('→ Vérifier que la cellule 4 s\'est terminée sans erreur')
else:
    size_mb = out_file.stat().st_size / (1024 * 1024)
    print(f'Fichier trouvé : {out_file.name}  ({size_mb:.1f} MB)')

    custos_src = Path(DRIVE_BASE) / 'CRS_CUSTOS.py'
    shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')

    result = subprocess.run(
        [sys.executable, '/content/CRS_CUSTOS.py',
         '--frigate', 'F04B', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
        capture_output=False
    )
    if result.returncode == 0:
        print('\n✓ CRS_CUSTOS — F04B check-in OK')
        print('='*52)
        print('PIPELINE PENTERACT DORN — CAMPAGNE TERMINÉE')
        print('='*52)
        print(f'→ {out_file.name} prêt pour upload YouTube')
    else:
        print('\n✗ CRS_CUSTOS — F04B check-in FAIL')